# Ethylene case study

This notebook demonstrates environmental and economic optimization of three
ethylene routes with optimex. All results remain provisional while proxy assumptions
are present.


## 1. Data and scenario

The study covers Europe from 2025 to 2050 and uses the
`REMIND-EU_SSP2-NDC` premise databases. Annual ethylene demand is constant at
`1e9 kg/a`. The routes are steam cracking, methanol-to-olefins supplied by DAC,
PEM electrolysis and CO2 hydrogenation, and electrochemical CO2 reduction with
aggregated product separation.

The use phase, ethylene product end-of-life and plant end-of-life are excluded.
The `eol=yes` inventory contribution retained in the eCO2R separation represents
oxidation of process by-products, not ethylene product end-of-life.


In [1]:
from datetime import datetime
from pathlib import Path

import bw2data as bd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyomo.environ as pyo
from bw_temporalis import TemporalDistribution
from IPython.display import display

from optimex.utils import (
    infer_construction_td_from_limits,
    infer_operation_td_from_limits,
)

PROJECT = "optimex_remind"
BIOSPHERE_DB = "ecoinvent-3.12-biosphere"
FOREGROUND_DB = "ethylene_case_study_foreground"
PREMISE_DBS = {
    2020: "ei312_REMIND-EU_SSP2_NDC_2020",
    2030: "ei312_REMIND-EU_SSP2_NDC_2030",
    2040: "ei312_REMIND-EU_SSP2_NDC_2040",
    2050: "ei312_REMIND-EU_SSP2_NDC_2050",
}

bd.projects.set_current(PROJECT)
required_databases = {BIOSPHERE_DB, *PREMISE_DBS.values()}
missing_databases = required_databases - set(bd.databases)
if missing_databases:
    raise ValueError(f"Missing databases: {sorted(missing_databases)}")

for year, database_name in PREMISE_DBS.items():
    bd.Database(database_name).metadata["representative_time"] = (
        datetime(year, 1, 1).isoformat()
    )


c:\Users\lucal\Brightway\optimex\.venv\Lib\site-packages\bw2calc\__init__.py:57: UserWarning: No fast sparse solver found
  warnings.warn("No fast sparse solver found")
c:\Users\lucal\Brightway\optimex\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.2) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


## 2. Model assumptions

The table collects the assumptions that control plant lifetimes, installation
scaling, existing capacity and discounting. Installation-edge amounts are copied
unchanged from their source inventories. Process lifetimes only control Optimex
vintage availability and replacement; they are not used to rescale inventory
coefficients. Final runs require the corrected `var_installation` semantics to
pass a multi-year static-LCA equivalence test. The brownfield capacities must
also be revalidated against that corrected variable definition.


In [2]:
assumptions = pd.DataFrame(
    [
        ("lifetime.steam_cracking", 50, "year", "PROXY", "ecoinvent 3.12 dataset documentation, p. 11", True),
        ("lifetime.dac", 15, "year", "PROXY", "methanol_and_iron blueprint", True),
        ("lifetime.pem", 8, "year", "PROXY", "methanol_and_iron stack lifetime", True),
        ("lifetime.co2_hydrogenation", 15, "year", "PROXY", "methanol_and_iron blueprint", True),
        ("lifetime.mto", 25, "year", "PROXY", "Chemical-plant proxy", True),
        ("lifetime.eco2r_reactor", 15, "year", "PROXY", "Emerging-technology proxy", True),
        ("lifetime.eco2r_separation", 15, "year", "PROXY", "Separation-plant proxy", True),
        (
            "installation.steam_factory",
            1.1516356618335166e-10,
            "unit/kg ethylene",
            "INVENTORY",
            "ecoinvent 3.12 inventory coefficient copied unchanged",
            False,
        ),
        (
            "installation.dac_system",
            1.25e-8,
            "unit/kg CO2",
            "INVENTORY",
            "disco2very inventory coefficient copied unchanged",
            False,
        ),
        (
            "installation.pem_stack",
            1.34989e-6,
            "unit/kg H2",
            "INVENTORY",
            "methanol_and_iron inventory coefficient copied unchanged",
            False,
        ),
        (
            "installation.pem_bop",
            3.37373e-7,
            "unit/kg H2",
            "INVENTORY",
            "methanol_and_iron inventory coefficient copied unchanged",
            False,
        ),
        (
            "installation.methanol_factory",
            3.5842e-12,
            "unit/kg methanol",
            "INVENTORY",
            "disco2very inventory coefficient copied unchanged",
            False,
        ),
        (
            "installation.mto_factory",
            3.584e-12,
            "unit/kg ethylene",
            "INVENTORY",
            "disco2very inventory coefficient copied unchanged",
            False,
        ),
        (
            "installation.eco2r_factory",
            7.32e-7,
            "kg factory/kg raw ethylene",
            "INVENTORY",
            "disco2very inventory coefficient copied unchanged",
            False,
        ),
        (
            "installation.eco2r_copper",
            7.4e-11,
            "kg/kg raw ethylene",
            "INVENTORY",
            "disco2very inventory coefficient copied unchanged",
            False,
        ),
        (
            "installation.separation_steel",
            3.017679e-6,
            "kg/kg ethylene",
            "INVENTORY",
            "disco2very inventory coefficient copied unchanged",
            False,
        ),
        ("brownfield.vintage_1", 2005, "year", "PROXY", "methanol_and_iron brownfield pattern", True),
        ("brownfield.vintage_2", 2015, "year", "PROXY", "methanol_and_iron brownfield pattern", True),
        ("brownfield.capacity_per_vintage", 0.5e9, "kg ethylene/a", "PROXY", "Two equal vintages", False),
        ("economics.discount_rate", 0.03, "real fraction/a", "PROXY", "basic_example_econ", True),
    ],
    columns=[
        "parameter", "value", "unit", "status", "basis", "replacement_needed"
    ],
).set_index("parameter")

PROCESS_NAMES = {
    "steam_cracking": "Steam cracking",
    "dac": "Direct air capture",
    "pem": "PEM electrolysis",
    "co2_hydrogenation": "CO2 hydrogenation to methanol",
    "mto": "Methanol-to-olefins",
    "eco2r_reactor": "Electrochemical CO2 reduction",
    "eco2r_separation": "eCO2R product separation",
}
LIFETIMES_YEARS = {
    process: int(assumptions.loc[f"lifetime.{process}", "value"])
    for process in PROCESS_NAMES
}
INSTALLATION_AMOUNTS = {
    key.removeprefix("installation."): float(row["value"])
    for key, row in assumptions.loc[
        assumptions.index.str.startswith("installation.")
    ].iterrows()
}

assumptions


,value,unit,status,basis,replacement_needed
parameter,,,,,
lifetime.steam_cracking,5.000000e+01,year,PROXY,"ecoinvent 3.12 dataset documentation, p. 11",True
lifetime.dac,1.500000e+01,year,PROXY,methanol_and_iron blueprint,True
lifetime.pem,8.000000e+00,year,PROXY,methanol_and_iron stack lifetime,True
lifetime.co2_hydrogenation,1.500000e+01,year,PROXY,methanol_and_iron blueprint,True
lifetime.mto,2.500000e+01,year,PROXY,Chemical-plant proxy,True
lifetime.eco2r_reactor,1.500000e+01,year,PROXY,Emerging-technology proxy,True
lifetime.eco2r_separation,1.500000e+01,year,PROXY,Separation-plant proxy,True
installation.steam_factory,1.151636e-10,unit/kg ethylene,INVENTORY,"ecoinvent 3.12 dataset documentation, p. 11; c...",False
installation.dac_system,1.250000e-08,unit/kg CO2,INVENTORY,disco2very inventory coefficient copied uncha...,False


## 3. Background inputs

All operating and installation inputs are selected directly from the 2020
premise support database. Their names, products, locations and units provide the
identity used by optimex to resolve the corresponding activities in later
support years.


In [3]:
REFERENCE_DB = PREMISE_DBS[2020]

steam_cracking_inventory = bd.get_node(
    database=REFERENCE_DB,
    name="unsaturated hydrocarbons production, steam cracking operation, average",
    product="ethylene",
    location="RER w/o RU",
    unit="kilogram",
)
electricity_mv = bd.get_node(
    database=REFERENCE_DB,
    name="market for electricity, medium voltage",
    product="electricity, medium voltage",
    location="DE",
    unit="kilowatt hour",
)
heat_pump = bd.get_node(
    database=REFERENCE_DB,
    name="heat production, at heat pump 30kW, allocation exergy",
    product="heat, central or small-scale, other than natural gas",
    location="Europe without Switzerland",
    unit="megajoule",
)
water_deionized = bd.get_node(
    database=REFERENCE_DB,
    name="market for water, deionised",
    product="water, deionised",
    location="Europe without Switzerland",
    unit="kilogram",
)
wastewater = bd.get_node(
    database=REFERENCE_DB,
    name="market for wastewater, unpolluted",
    product="wastewater, unpolluted",
    location="RoW",
    unit="cubic meter",
)
cooling = bd.get_node(
    database=REFERENCE_DB,
    name="market for cooling energy",
    product="cooling energy",
    location="GLO",
    unit="megajoule",
)
cooling_minus_15 = bd.get_node(
    database=REFERENCE_DB,
    name="market for cooling energy, at -15 °C",
    product="cooling energy, at -15 °C",
    location="GLO",
    unit="megajoule",
)
cooling_minus_25 = bd.get_node(
    database=REFERENCE_DB,
    name="cooling energy production, at -25 °C, propylene compression refrigeration system 1 MW",
    product="cooling energy, at -25 °C",
    location="GLO",
    unit="megajoule",
)
cooling_minus_45 = bd.get_node(
    database=REFERENCE_DB,
    name="market for cooling energy, at -45 °C",
    product="cooling energy, at -45 °C",
    location="GLO",
    unit="megajoule",
)
cooling_minus_55 = bd.get_node(
    database=REFERENCE_DB,
    name="market for cooling energy, at -55 °C",
    product="cooling energy, at -55 °C",
    location="GLO",
    unit="megajoule",
)
cooling_minus_100 = bd.get_node(
    database=REFERENCE_DB,
    name="market for cooling energy, at -100 °C",
    product="cooling energy, at -100 °C",
    location="GLO",
    unit="megajoule",
)
industrial_heat = bd.get_node(
    database=REFERENCE_DB,
    name="market for heat, district or industrial, natural gas",
    product="heat, district or industrial, natural gas",
    location="Europe without Switzerland",
    unit="megajoule",
)
adsorbent_proxy = bd.get_node(
    database=REFERENCE_DB,
    name="market for activated carbon, granular",
    product="activated carbon, granular",
    location="GLO",
    unit="kilogram",
)
spent_adsorbent_treatment = bd.get_node(
    database=REFERENCE_DB,
    name="treatment of spent anion exchange resin from potable water production, municipal incineration",
    product="spent anion exchange resin from potable water production",
    location="GLO",
    unit="kilogram",
)

chemical_factory_organics = bd.get_node(
    database=REFERENCE_DB,
    name="chemical factory construction, organics",
    product="chemical factory, organics",
    location="RER",
    unit="unit",
)
chemical_factory = bd.get_node(
    database=REFERENCE_DB,
    name="chemical factory construction",
    product="chemical factory",
    location="RER",
    unit="kilogram",
)
dac_system = bd.get_node(
    database=REFERENCE_DB,
    name="direct air capture system, solvent-based, 1MtCO2",
    product="direct air capture system",
    location="RER",
    unit="unit",
)
pem_stack = bd.get_node(
    database=REFERENCE_DB,
    name="electrolyzer production, 1MWe, PEM, Stack",
    product="electrolyzer, 1MWe, PEM, Stack",
    location="RER",
    unit="unit",
)
pem_bop = bd.get_node(
    database=REFERENCE_DB,
    name="electrolyzer production, 1MWe, PEM, Balance of Plant",
    product="electrolyzer, 1MWe, PEM, Balance of Plant",
    location="RER",
    unit="unit",
)
methanol_facility = bd.get_node(
    database=REFERENCE_DB,
    name="methanol production facility, construction",
    product="methanol production facility, construction",
    location="RER",
    unit="unit",
)
low_alloyed_steel = bd.get_node(
    database=REFERENCE_DB,
    name="market for steel, low-alloyed",
    product="steel, low-alloyed",
    location="GLO",
    unit="kilogram",
)
copper_cathode = bd.get_node(
    database=REFERENCE_DB,
    name="market for copper, cathode",
    product="copper, cathode",
    location="GLO",
    unit="kilogram",
)
co2_fossil = bd.get_node(
    database=BIOSPHERE_DB,
    name="Carbon dioxide, fossil",
    categories=("air",),
)


## 4. Products and processes

Five foreground products connect seven decision processes. Operation exchanges
scale with operation, while installation exchanges scale with new installations.
All exchange amounts retain the per-reference-product basis of their source
inventories.


In [4]:
if FOREGROUND_DB in bd.databases:
    del bd.databases[FOREGROUND_DB]
foreground = bd.Database(FOREGROUND_DB)
foreground.register()

ethylene = foreground.new_node(
    code="ethylene",
    name="Ethylene",
    unit="kilogram",
    type=bd.labels.product_node_default,
)
ethylene.save()

captured_co2 = foreground.new_node(
    code="captured_co2",
    name="Captured carbon dioxide",
    unit="kilogram",
    type=bd.labels.product_node_default,
)
captured_co2.save()

hydrogen = foreground.new_node(
    code="hydrogen",
    name="Hydrogen",
    unit="kilogram",
    type=bd.labels.product_node_default,
)
hydrogen.save()

methanol = foreground.new_node(
    code="methanol",
    name="Methanol",
    unit="kilogram",
    type=bd.labels.product_node_default,
)
methanol.save()

eco2r_raw_ethylene = foreground.new_node(
    code="eco2r_raw_ethylene",
    name="Ethylene before eCO2R separation",
    unit="kilogram",
    type=bd.labels.product_node_default,
)
eco2r_raw_ethylene.save()


### Steam cracking

The conventional route exposes the direct operating inputs and direct
biosphere exchanges of the ecoinvent steam-cracking dataset. This places it
at the same foreground depth as the alternative routes. The biosphere table
contains only exchanges attached directly to the steam-cracking activity;
Brightway still calculates cumulative supply-chain emissions through the
technosphere inputs. The organic chemical factory is represented separately
as an installation using the unchanged ecoinvent exchange amount. The process
lifetime remains a separate Optimex assumption.


In [5]:
steam = foreground.new_node(
    code="steam_cracking",
    name=PROCESS_NAMES["steam_cracking"],
    location="RER w/o RU",
    type=bd.labels.process_node_default,
    operation_time_limits=(0, LIFETIMES_YEARS["steam_cracking"] - 1),
)
steam.save()

steam_cracking_operating_amounts = [
    ("market for butane", 0.19711504876613617),
    ("market for compressed air, 700 kPa gauge", 0.014663908630609512),
    ("market for diesel", 0.04867038503289223),
    ("market for ethane", 0.05475417897105217),
    ("market for hazardous waste, for incineration", -0.003802229417487979),
    ("market for hazardous waste, for underground deposit", -0.0010579951340332627),
    ("market for inert waste, for final disposal", -0.00037626258563250303),
    ("market for methanol", 0.00023348795366473496),
    ("market for naphtha", 0.7787261009216309),
    ("market for natural gas liquids", 0.02676870860159397),
    ("market for nitrogen, liquid", 0.011272253468632698),
    ("market for propane", 0.09247373044490814),
    ("market for refinery gas", 0.018251389265060425),
    ("market for sodium hydroxide, without water, in 50% solution state", 0.004936636425554752),
    ("market for wastewater, average", -0.0006797355017624795),
    ("market for water, deionised", 2.186876058578491),
    ("market group for electricity, medium voltage", 0.1206742525100708),
]
assert len(steam_cracking_operating_amounts) == 17

source_technosphere = list(steam_cracking_inventory.technosphere())
steam_cracking_operating_inputs = []
for activity_name, amount in steam_cracking_operating_amounts:
    matches = [
        exchange for exchange in source_technosphere
        if exchange.input.get("name") == activity_name
    ]
    if len(matches) != 1:
        raise ValueError(
            f"Expected one steam-cracking input named '{activity_name}', "
            f"found {len(matches)}."
        )
    source_exchange = matches[0]
    if not np.isclose(float(source_exchange["amount"]), amount):
        raise ValueError(
            f"Unexpected amount for '{activity_name}': "
            f"{source_exchange['amount']} instead of {amount}."
        )
    steam_cracking_operating_inputs.append(
        {"input": source_exchange.input, "amount": amount}
    )

steam_cracking_input_table = pd.DataFrame(
    [
        {
            "name": item["input"].get("name"),
            "product": item["input"].get("reference product"),
            "location": item["input"].get("location"),
            "unit": item["input"].get("unit"),
            "amount per kg ethylene": item["amount"],
        }
        for item in steam_cracking_operating_inputs
    ]
)

steam_cracking_biosphere_exchanges = list(steam_cracking_inventory.biosphere())
assert len(steam_cracking_biosphere_exchanges) == 44
steam_cracking_biosphere_table = pd.DataFrame(
    [
        {
            "name": exchange.input.get("name"),
            "categories": exchange.input.get("categories"),
            "unit": exchange.input.get("unit"),
            "amount per kg ethylene": float(exchange["amount"]),
        }
        for exchange in steam_cracking_biosphere_exchanges
    ]
)

display(steam_cracking_input_table)
display(steam_cracking_biosphere_table)

steam.new_edge(
    input=ethylene,
    amount=1.0,
    type=bd.labels.production_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(steam),
).save()
for item in steam_cracking_operating_inputs:
    steam.new_edge(
        input=item["input"],
        amount=item["amount"],
        type=bd.labels.consumption_edge_default,
        operation=True,
        temporal_distribution=infer_operation_td_from_limits(steam),
    ).save()

for exchange in steam_cracking_biosphere_exchanges:
    steam.new_edge(
        input=exchange.input,
        amount=float(exchange["amount"]),
        type=bd.labels.biosphere_edge_default,
        operation=True,
        temporal_distribution=infer_operation_td_from_limits(steam),
    ).save()
steam.new_edge(
    input=chemical_factory_organics,
    amount=INSTALLATION_AMOUNTS["steam_factory"],
    type=bd.labels.consumption_edge_default,
    operation=False,
    temporal_distribution=infer_construction_td_from_limits(steam),
).save()


,name,product,location,unit,amount per kg ethylene
0,market for butane,butane,RoW,kilogram,0.197115
1,"market for compressed air, 700 kPa gauge","compressed air, 700 kPa gauge",RER,cubic meter,0.014664
2,market for diesel,diesel,DEU,kilogram,0.048670
3,market for ethane,ethane,RoW,kilogram,0.054754
4,"market for hazardous waste, for incineration","hazardous waste, for incineration",CH,kilogram,-0.003802
5,"market for hazardous waste, for underground de...","hazardous waste, for underground deposit",RER,kilogram,-0.001058
6,"market for inert waste, for final disposal","inert waste, for final disposal",CH,kilogram,-0.000376
7,market for methanol,methanol,RER w/o RU,kilogram,0.000233
8,market for naphtha,naphtha,RER,kilogram,0.778726
9,market for natural gas liquids,natural gas liquids,RoW,kilogram,0.026769


,name,categories,unit,amount per kg ethylene
0,Acetylene,"(air,)",kilogram,5.203157e-07
1,Benzene,"(air,)",kilogram,1.034685e-05
2,Butadiene,"(air,)",kilogram,5.024763e-06
3,Butane,"(air,)",kilogram,1.486616e-07
4,Butene,"(air,)",kilogram,5.203157e-06
5,"Carbon dioxide, fossil","(air,)",kilogram,7.124258e-01
6,"Carbon monoxide, fossil","(air,)",kilogram,2.126902e-04
7,Chlorodifluoromethane,"(air,)",kilogram,1.486616e-08
8,Ethane,"(air,)",kilogram,7.314152e-06
9,Ethylene,"(air,)",kilogram,2.009905e-05


### Direct air capture

DAC produces the shared captured-CO2 product. Electricity, heat, the adsorbent
proxy, spent-adsorbent treatment and atmospheric CO2 uptake are operating
flows. The DAC system is installed separately using a provisional coefficient.


In [6]:
dac = foreground.new_node(
    code="dac",
    name=PROCESS_NAMES["dac"],
    location="RER",
    type=bd.labels.process_node_default,
    operation_time_limits=(0, LIFETIMES_YEARS["dac"] - 1),
)
dac.save()

dac.new_edge(
    input=captured_co2,
    amount=1.0,
    type=bd.labels.production_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(dac),
).save()
dac.new_edge(
    input=electricity_mv,
    amount=0.7,
    type=bd.labels.consumption_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(dac),
).save()
dac.new_edge(
    input=heat_pump,
    amount=4.7,
    type=bd.labels.consumption_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(dac),
).save()
dac.new_edge(
    input=adsorbent_proxy,
    amount=0.0075,
    type=bd.labels.consumption_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(dac),
).save()
dac.new_edge(
    input=spent_adsorbent_treatment,
    amount=-0.0075,
    type=bd.labels.consumption_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(dac),
).save()
dac.new_edge(
    input=co2_fossil,
    amount=-1.0,
    type=bd.labels.biosphere_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(dac),
).save()
dac.new_edge(
    input=dac_system,
    amount=INSTALLATION_AMOUNTS["dac_system"],
    type=bd.labels.consumption_edge_default,
    operation=False,
    temporal_distribution=infer_construction_td_from_limits(dac),
).save()


### PEM electrolysis

PEM electrolysis produces hydrogen from electricity and deionized water. Stack
and balance-of-plant exchange amounts are copied unchanged from the
`methanol_and_iron` inventory. Their source lifetimes are not needed for
installation scaling.


In [7]:
pem = foreground.new_node(
    code="pem",
    name=PROCESS_NAMES["pem"],
    location="RER",
    type=bd.labels.process_node_default,
    operation_time_limits=(0, LIFETIMES_YEARS["pem"] - 1),
)
pem.save()

pem.new_edge(
    input=hydrogen,
    amount=1.0,
    type=bd.labels.production_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(pem),
).save()
pem.new_edge(
    input=electricity_mv,
    amount=56.00509259,
    type=bd.labels.consumption_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(pem),
).save()
pem.new_edge(
    input=water_deionized,
    amount=8.936011905,
    type=bd.labels.consumption_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(pem),
).save()
pem.new_edge(
    input=wastewater,
    amount=-2.2e-5,
    type=bd.labels.consumption_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(pem),
).save()
pem.new_edge(
    input=pem_stack,
    amount=INSTALLATION_AMOUNTS["pem_stack"],
    type=bd.labels.consumption_edge_default,
    operation=False,
    temporal_distribution=infer_construction_td_from_limits(pem),
).save()
pem.new_edge(
    input=pem_bop,
    amount=INSTALLATION_AMOUNTS["pem_bop"],
    type=bd.labels.consumption_edge_default,
    operation=False,
    temporal_distribution=infer_construction_td_from_limits(pem),
).save()


### CO2 hydrogenation

CO2 hydrogenation combines the shared captured-CO2 and hydrogen products to
produce methanol. Electricity and wastewater are operating flows; the methanol
facility is a separate installation with its unchanged inventory amount.


In [8]:
co2_hydrogenation = foreground.new_node(
    code="co2_hydrogenation",
    name=PROCESS_NAMES["co2_hydrogenation"],
    location="RER",
    type=bd.labels.process_node_default,
    operation_time_limits=(0, LIFETIMES_YEARS["co2_hydrogenation"] - 1),
)
co2_hydrogenation.save()

co2_hydrogenation.new_edge(
    input=methanol,
    amount=1.0,
    type=bd.labels.production_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(co2_hydrogenation),
).save()
co2_hydrogenation.new_edge(
    input=electricity_mv,
    amount=1.32878456384964,
    type=bd.labels.consumption_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(co2_hydrogenation),
).save()
co2_hydrogenation.new_edge(
    input=captured_co2,
    amount=1.435820454,
    type=bd.labels.consumption_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(co2_hydrogenation),
).save()
co2_hydrogenation.new_edge(
    input=hydrogen,
    amount=0.197319687,
    type=bd.labels.consumption_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(co2_hydrogenation),
).save()
co2_hydrogenation.new_edge(
    input=wastewater,
    amount=-0.000562265917602996,
    type=bd.labels.consumption_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(co2_hydrogenation),
).save()
co2_hydrogenation.new_edge(
    input=methanol_facility,
    amount=INSTALLATION_AMOUNTS["methanol_factory"],
    type=bd.labels.consumption_edge_default,
    operation=False,
    temporal_distribution=infer_construction_td_from_limits(co2_hydrogenation),
).save()


### Methanol-to-olefins

MTO converts methanol to ethylene using the mass-allocated disco2very inventory
values. The -30 °C and -75 °C cooling duties use the available -25 °C and
-100 °C premise cooling processes as visible operating proxies. The organic
chemical factory is installed separately.


In [9]:
mto = foreground.new_node(
    code="mto",
    name=PROCESS_NAMES["mto"],
    location="RER",
    type=bd.labels.process_node_default,
    operation_time_limits=(0, LIFETIMES_YEARS["mto"] - 1),
)
mto.save()

mto.new_edge(
    input=ethylene,
    amount=1.0,
    type=bd.labels.production_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(mto),
).save()
mto.new_edge(
    input=electricity_mv,
    amount=0.1428058844,
    type=bd.labels.consumption_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(mto),
).save()
mto.new_edge(
    input=cooling,
    amount=0.17224,
    type=bd.labels.consumption_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(mto),
).save()
mto.new_edge(
    input=cooling_minus_25,
    amount=0.4304 + 0.9216,
    type=bd.labels.consumption_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(mto),
).save()
mto.new_edge(
    input=cooling_minus_100,
    amount=0.21528,
    type=bd.labels.consumption_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(mto),
).save()
mto.new_edge(
    input=wastewater,
    amount=-0.0012778376,
    type=bd.labels.consumption_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(mto),
).save()
mto.new_edge(
    input=methanol,
    amount=2.3920583664,
    type=bd.labels.consumption_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(mto),
).save()
mto.new_edge(
    input=chemical_factory_organics,
    amount=INSTALLATION_AMOUNTS["mto_factory"],
    type=bd.labels.consumption_edge_default,
    operation=False,
    temporal_distribution=infer_construction_td_from_limits(mto),
).save()


### Electrochemical CO2 reduction

The eCO2R reactor produces one kilogram of raw ethylene for the downstream
separation process. Electricity, water and captured CO2 are operating inputs;
the chemical factory and copper electrode are provisional installations.


In [10]:
eco2r = foreground.new_node(
    code="eco2r_reactor",
    name=PROCESS_NAMES["eco2r_reactor"],
    location="DE",
    type=bd.labels.process_node_default,
    operation_time_limits=(0, LIFETIMES_YEARS["eco2r_reactor"] - 1),
)
eco2r.save()

eco2r.new_edge(
    input=eco2r_raw_ethylene,
    amount=1.0,
    type=bd.labels.production_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(eco2r),
).save()
eco2r.new_edge(
    input=electricity_mv,
    amount=74.878,
    type=bd.labels.consumption_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(eco2r),
).save()
eco2r.new_edge(
    input=water_deionized,
    amount=4.4249,
    type=bd.labels.consumption_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(eco2r),
).save()
eco2r.new_edge(
    input=captured_co2,
    amount=9.229,
    type=bd.labels.consumption_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(eco2r),
).save()
eco2r.new_edge(
    input=chemical_factory,
    amount=INSTALLATION_AMOUNTS["eco2r_factory"],
    type=bd.labels.consumption_edge_default,
    operation=False,
    temporal_distribution=infer_construction_td_from_limits(eco2r),
).save()
eco2r.new_edge(
    input=copper_cathode,
    amount=INSTALLATION_AMOUNTS["eco2r_copper"],
    type=bd.labels.consumption_edge_default,
    operation=False,
    temporal_distribution=infer_construction_td_from_limits(eco2r),
).save()


### eCO2R product separation

Vapor-liquid separation, oxygen removal, amine washing, temperature-swing
adsorption and cryogenic separation are represented by one aggregated process.
The `6.091081 kg CO2/kg ethylene` emission is oxidation of process by-products,
not ethylene product end-of-life. Separator steel is a provisional installation.


In [11]:
separation = foreground.new_node(
    code="eco2r_separation",
    name=PROCESS_NAMES["eco2r_separation"],
    location="DE",
    type=bd.labels.process_node_default,
    operation_time_limits=(0, LIFETIMES_YEARS["eco2r_separation"] - 1),
)
separation.save()

separation.new_edge(
    input=ethylene,
    amount=1.0,
    type=bd.labels.production_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(separation),
).save()
separation.new_edge(
    input=eco2r_raw_ethylene,
    amount=1.0,
    type=bd.labels.consumption_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(separation),
).save()
separation.new_edge(
    input=electricity_mv,
    amount=0.133897031 + 0.5324327,
    type=bd.labels.consumption_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(separation),
).save()
separation.new_edge(
    input=cooling,
    amount=0.010118339 + 3.06967977 + 2.0085,
    type=bd.labels.consumption_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(separation),
).save()
separation.new_edge(
    input=industrial_heat,
    amount=28.487337 + 3.079241549,
    type=bd.labels.consumption_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(separation),
).save()
separation.new_edge(
    input=cooling_minus_15,
    amount=0.17912,
    type=bd.labels.consumption_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(separation),
).save()
separation.new_edge(
    input=cooling_minus_25,
    amount=0.057779,
    type=bd.labels.consumption_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(separation),
).save()
separation.new_edge(
    input=cooling_minus_45,
    amount=0.11556,
    type=bd.labels.consumption_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(separation),
).save()
separation.new_edge(
    input=cooling_minus_55,
    amount=0.057779,
    type=bd.labels.consumption_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(separation),
).save()
separation.new_edge(
    input=cooling_minus_100,
    amount=0.60715,
    type=bd.labels.consumption_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(separation),
).save()
separation.new_edge(
    input=wastewater,
    amount=-0.000763598,
    type=bd.labels.consumption_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(separation),
).save()
separation.new_edge(
    input=co2_fossil,
    amount=2.361481 + 3.7296,
    type=bd.labels.biosphere_edge_default,
    operation=True,
    temporal_distribution=infer_operation_td_from_limits(separation),
).save()
separation.new_edge(
    input=low_alloyed_steel,
    amount=INSTALLATION_AMOUNTS["separation_steel"],
    type=bd.labels.consumption_edge_default,
    operation=False,
    temporal_distribution=infer_construction_td_from_limits(separation),
).save()

processes = {
    "steam_cracking": steam,
    "dac": dac,
    "pem": pem,
    "co2_hydrogenation": co2_hydrogenation,
    "mto": mto,
    "eco2r_reactor": eco2r,
    "eco2r_separation": separation,
}


## 5. Demand and existing capacity

The system must supply `1 Mt` of ethylene per year. Two equal steam-cracking
vintages provide the full initial annual capacity; their installation years are
visible proxies to be reviewed with the final lifetime assumptions.


In [12]:
SYSTEM_YEARS = np.arange(2025, 2051)
ANNUAL_ETHYLENE_DEMAND_KG = 1e9
ethylene_demand = TemporalDistribution(
    date=np.array(
        [datetime(int(year), 1, 1).isoformat() for year in SYSTEM_YEARS],
        dtype="datetime64[s]",
    ),
    amount=np.full(len(SYSTEM_YEARS), ANNUAL_ETHYLENE_DEMAND_KG),
)
functional_demand = {ethylene: ethylene_demand}

# PR #61 semantics: existing_capacity stores process units. One steam-cracker
# unit delivers the 1 kg production exchange over its 50-year lifetime.
steam_cracking_lifetime_years = int(LIFETIMES_YEARS["steam_cracking"])
steam_cracking_annual_output_per_unit_kg = 1.0 / steam_cracking_lifetime_years
brownfield_annual_capacity_per_vintage_kg = float(
    assumptions.loc["brownfield.capacity_per_vintage", "value"]
)
brownfield_units_per_vintage = (
    brownfield_annual_capacity_per_vintage_kg
    / steam_cracking_annual_output_per_unit_kg
)

existing_capacities = {
    (
        "steam_cracking",
        int(assumptions.loc["brownfield.vintage_1", "value"]),
    ): brownfield_units_per_vintage,
    (
        "steam_cracking",
        int(assumptions.loc["brownfield.vintage_2", "value"]),
    ): brownfield_units_per_vintage,
}

brownfield_table = pd.DataFrame(
    [
        {
            "process": PROCESS_NAMES[process],
            "installation_year": year,
            "installed_process_units": units,
            "annual_capacity_kg_per_year": (
                units * steam_cracking_annual_output_per_unit_kg
            ),
        }
        for (process, year), units in existing_capacities.items()
    ]
)
assert np.allclose(
    brownfield_table["annual_capacity_kg_per_year"],
    brownfield_annual_capacity_per_vintage_kg,
)
assert np.isclose(
    brownfield_table["annual_capacity_kg_per_year"].sum(),
    2 * brownfield_annual_capacity_per_vintage_kg,
)
brownfield_table


,process,installation_year,capacity_kg_per_year
0,Steam cracking,2005,500000000.0
1,Steam cracking,2015,500000000.0


## 6. Market-price inputs

The versioned CSV contains price trajectories and source metadata for direct
operating and installation purchases. The table below checks coverage against
the foreground and then attaches the prices to the four premise support
databases. Missing prices retain the existing warning-based, user-responsibility
handling.


In [13]:
from optimex.economics import set_market_prices

flow_metadata = {}
flow_classes = {}
for process_code, process_node in processes.items():
    for exchange in process_node.technosphere():
        if exchange.input.get("database") == FOREGROUND_DB:
            continue
        product_name = (
            exchange.input.get("reference product")
            or exchange.input.get("product")
        )
        identity = (
            exchange.input.get("name"),
            product_name,
            exchange.input.get("location"),
            exchange.input.get("unit"),
        )
        flow_metadata[identity] = {
            "name": identity[0],
            "product": identity[1],
            "location": identity[2],
            "unit": identity[3],
        }
        flow_classes.setdefault(identity, set()).add(
            "op" if exchange.get("operation") else "cap"
        )

required_cost_flows = pd.DataFrame(
    [
        {
            **flow_metadata[identity],
            "cost_class": (
                "cap_and_op"
                if classes == {"cap", "op"}
                else next(iter(classes))
            ),
        }
        for identity, classes in flow_classes.items()
    ]
).sort_values(["cost_class", "name"], ignore_index=True)

repo_root = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
cost_csv = (
    repo_root / "notebooks" / "data" / "ethylene_case_study" / "cost_inputs.csv"
)
if not cost_csv.exists():
    raise FileNotFoundError(f"Missing versioned cost input file: {cost_csv}")

cost_inputs = pd.read_csv(cost_csv)
identity_columns = ["name", "product", "location", "unit"]
csv_coverage = cost_inputs[
    identity_columns + ["status"]
].drop_duplicates(identity_columns)
cost_coverage = required_cost_flows.merge(
    csv_coverage,
    on=identity_columns,
    how="left",
)
missing_prices = cost_coverage[cost_coverage["status"].isna()]
if not missing_prices.empty:
    print("WARNING: Missing market-price rows for these foreground flows:")
    display(missing_prices)

set_market_prices(
    price_data=cost_inputs,
    background_databases=PREMISE_DBS,
    product_col="product",
    unit_col="unit",
    strict=False,
)

display(cost_coverage)
cost_inputs.groupby("status").size().rename("CSV rows").to_frame()


,name,product,location,unit,cost_class,status
0,chemical factory construction,chemical factory,RER,kilogram,cap,PROXY
1,"chemical factory construction, organics","chemical factory, organics",RER,unit,cap,PROXY
2,"direct air capture system, solvent-based, 1MtCO2",direct air capture system,RER,unit,cap,PROXY
3,"electrolyzer production, 1MWe, PEM, Balance of...","electrolyzer, 1MWe, PEM, Balance of Plant",RER,unit,cap,PROXY
4,"electrolyzer production, 1MWe, PEM, Stack","electrolyzer, 1MWe, PEM, Stack",RER,unit,cap,PROXY
5,"market for copper, cathode","copper, cathode",GLO,kilogram,cap,PROXY
6,"market for steel, low-alloyed","steel, low-alloyed",GLO,kilogram,cap,PROXY
7,"methanol production facility, construction","methanol production facility, construction",RER,unit,cap,PROXY
8,"cooling energy production, at -25 °C, propylen...","cooling energy, at -25 °C",GLO,megajoule,op,PROXY
9,"heat production, at heat pump 30kW, allocation...","heat, central or small-scale, other than natur...",Europe without Switzerland,megajoule,op,PROXY


,CSV rows
status,
PLACEHOLDER,68
PROXY,84


## 7. Time-explicit LCA data

The foreground and annual demand are converted into the tensors used by
optimex. The resulting operating and installation cost-flow lists are shown as
the final price-input audit.


In [14]:
from optimex import converter, lca_processor

METHOD_CLIMATE_CHANGE = (
    "IPCC 2021",
    "climate change",
    "GWP 100a, incl. H and bio CO2",
)
lca_config = lca_processor.LCAConfig(
    demand=functional_demand,
    temporal={
        "start_date": datetime(2025, 1, 1),
        "temporal_resolution": "year",
        "time_horizon": 100,
        "database_dates": {
            database_name: datetime(year, 1, 1)
            for year, database_name in PREMISE_DBS.items()
        },
    },
    characterization_methods=[
        {
            "category_name": "climate_change",
            "brightway_method": METHOD_CLIMATE_CHANGE,
        }
    ],
)
lca_data_processor = lca_processor.LCADataProcessor(
    lca_config,
    foreground_db_name=FOREGROUND_DB,
)


def processor_cost_flow_table(processor):
    rows = []
    all_codes = (
        processor.cost_relevant_cap_flows
        | processor.cost_relevant_op_flows
    )
    for flow_code in sorted(all_codes):
        metadata = processor.intermediate_flows[flow_code]
        rows.append(
            {
                "flow_code": flow_code,
                **metadata,
                "cost_class": (
                    "cap_and_op"
                    if flow_code in processor.cost_relevant_cap_flows
                    and flow_code in processor.cost_relevant_op_flows
                    else "cap"
                    if flow_code in processor.cost_relevant_cap_flows
                    else "op"
                ),
            }
        )
    return pd.DataFrame(rows)


cost_relevant_flows = processor_cost_flow_table(lca_data_processor)
csv_identities = cost_inputs[
    ["name", "product", "location", "unit", "status"]
].drop_duplicates()
cost_relevant_flows = cost_relevant_flows.merge(
    csv_identities,
    on=["name", "product", "location", "unit"],
    how="left",
)

print(
    "cost_relevant_op_flows:",
    len(lca_data_processor.cost_relevant_op_flows),
)
print(
    "cost_relevant_cap_flows:",
    len(lca_data_processor.cost_relevant_cap_flows),
)
display(
    cost_relevant_flows[
        ["cost_class", "name", "product", "location", "unit", "status"]
    ].sort_values(["cost_class", "name"], ignore_index=True)
)
cost_relevant_flows.groupby("status", dropna=False).size().rename(
    "cost-relevant flows"
).to_frame()


2026-07-28 11:37:28.841 | INFO     | optimex.lca_processor:_parse_demand:460 - Identified demand in system time range of %s for products %s
2026-07-28 11:37:28.935 | INFO     | optimex.lca_processor:_construct_foreground_tensors:699 - Constructed foreground tensors.
2026-07-28 11:37:28.938 | INFO     | optimex.lca_processor:log_tensor_dimensions:694 - Technosphere (external) shape: (7 processes, 38 flows, 50 years) with 1263 total entries.
2026-07-28 11:37:28.939 | INFO     | optimex.lca_processor:log_tensor_dimensions:694 - Internal demand shape: (4 processes, 4 flows, 25 years) with 85 total entries.
2026-07-28 11:37:28.940 | INFO     | optimex.lca_processor:log_tensor_dimensions:694 - Biosphere shape: (3 processes, 44 flows, 50 years) with 2230 total entries.
2026-07-28 11:37:28.943 | INFO     | optimex.lca_processor:log_tensor_dimensions:694 - Production shape: (7 processes, 5 flows, 50 years) with 143 total entries.
2026-07-28 11:38:19.297 | INFO     | optimex.lca_processor:_calcu

cost_relevant_op_flows: 30
cost_relevant_cap_flows: 8


,cost_class,name,product,location,unit,status
0,cap,chemical factory construction,chemical factory,RER,kilogram,PROXY
1,cap,"chemical factory construction, organics","chemical factory, organics",RER,unit,PROXY
2,cap,"direct air capture system, solvent-based, 1MtCO2",direct air capture system,RER,unit,PROXY
3,cap,"electrolyzer production, 1MWe, PEM, Balance of...","electrolyzer, 1MWe, PEM, Balance of Plant",RER,unit,PROXY
4,cap,"electrolyzer production, 1MWe, PEM, Stack","electrolyzer, 1MWe, PEM, Stack",RER,unit,PROXY
5,cap,"market for copper, cathode","copper, cathode",GLO,kilogram,PROXY
6,cap,"market for steel, low-alloyed","steel, low-alloyed",GLO,kilogram,PROXY
7,cap,"methanol production facility, construction","methanol production facility, construction",RER,unit,PROXY
8,op,"cooling energy production, at -25 °C, propylen...","cooling energy, at -25 °C",GLO,megajoule,PROXY
9,op,"heat production, at heat pump 30kW, allocation...","heat, central or small-scale, other than natur...",Europe without Switzerland,megajoule,PROXY


,cost-relevant flows
status,
PLACEHOLDER,17
PROXY,21


In [ ]:
manager = converter.ModelInputManager()
optimization_inputs = manager.parse_from_lca_processor(lca_data_processor)
scenario_inputs = manager.override(
    existing_capacity=existing_capacities,
    vintage_improvements=None,
    discount_rate=float(
        assumptions.loc["economics.discount_rate", "value"]
    ),
    discount_reference_year=2025,
)

In [ ]:
manager.save("data/2026-07-18_model_inputs_ethylene.json")

## 8. Optimization scenarios

The first scenario minimizes cumulative climate-change impact. The second
minimizes discounted total cost without a CO2 price. Both use the same demand,
technology assumptions and existing steam-cracking capacity.


In [ ]:
from optimex import optimizer

SOLVER = "highs"


def solve_scenario(name, objective):
    model = optimizer.create_model(
        scenario_inputs.model_copy(deep=True),
        name=name,
        objective_category="climate_change",
        objective=objective,
    )
    return optimizer.solve_model(
        model,
        solver_name=SOLVER,
        tee=False,
    )


climate_model, climate_objective, climate_results = solve_scenario(
    "ethylene_climate_minimum",
    "environmental",
)
cost_model, cost_objective, cost_results = solve_scenario(
    "ethylene_cost_minimum_without_co2_price",
    "cost",
)


## 9. Results

The summary reports both objective values. The plots compare ethylene
production by route and newly installed annual capacity for the two scenarios.
Cost results are provisional while proxy or placeholder inputs remain.


In [ ]:
from optimex import postprocessing


def cumulative_climate_impact(model):
    return (
        pyo.value(model.total_impact["climate_change"])
        * model.scales["foreground"]
        * model.scales["characterization"]["climate_change"]
    )


result_summary = pd.DataFrame(
    [
        {
            "scenario": "Climate minimum",
            "cumulative_climate_kg_CO2eq": climate_objective,
            "discounted_cost_EUR_2025": pyo.value(climate_model.total_cost),
        },
        {
            "scenario": "Cost minimum without CO2 price",
            "cumulative_climate_kg_CO2eq": cumulative_climate_impact(cost_model),
            "discounted_cost_EUR_2025": cost_objective,
        },
    ]
).set_index("scenario")
result_summary


In [ ]:
plot_colors = {
    "steam_cracking": "#555555",
    "dac": "#3B82A0",
    "pem": "#2D9C7A",
    "co2_hydrogenation": "#73A942",
    "mto": "#B7A12A",
    "eco2r_reactor": "#4C78A8",
    "eco2r_separation": "#7B6FD0",
}


def scenario_tables(model):
    processor = postprocessing.PostProcessor(model)
    production = processor.get_production()
    ethylene_columns = [
        column for column in production.columns if column[1] == "ethylene"
    ]
    route_production = production[ethylene_columns].copy() / 1e9
    route_production.columns = [column[0] for column in ethylene_columns]
    route_production = route_production.loc[
        :, (route_production.abs() > 1e-9).any(axis=0)
    ]

    installation = processor.get_installation().copy() / 1e9
    installation = installation.loc[
        :, (installation.abs() > 1e-9).any(axis=0)
    ]
    return route_production, installation


scenario_models = {
    "Climate minimum": climate_model,
    "Cost minimum without CO2 price": cost_model,
}
fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex="col")

for column_index, (title, model) in enumerate(scenario_models.items()):
    production, installation = scenario_tables(model)
    production.rename(columns=PROCESS_NAMES).plot.area(
        ax=axes[0, column_index],
        stacked=True,
        color=[plot_colors[key] for key in production.columns],
        linewidth=0,
    )
    axes[0, column_index].set_title(title)
    axes[0, column_index].set_ylabel("Ethylene production [Mt/a]")
    axes[0, column_index].set_ylim(bottom=0)

    installation.rename(columns=PROCESS_NAMES).plot.bar(
        ax=axes[1, column_index],
        stacked=True,
        color=[plot_colors[key] for key in installation.columns],
        width=0.85,
    )
    axes[1, column_index].set_ylabel("New capacity [Mt reference product/a]")
    axes[1, column_index].set_xlabel("Installation year")
    axes[1, column_index].tick_params(axis="x", rotation=60)

for axis in axes.flat:
    axis.grid(axis="y", alpha=0.25)
    axis.legend(fontsize=8, frameon=False)

plt.tight_layout()
plt.show()


## 10. Interpretation boundary

All assumptions marked `replacement_needed=True` and all `PROXY` or
`PLACEHOLDER` prices must be reviewed before the results are used in the
Bachelor thesis. Installation-edge amounts remain unchanged from their source
inventories; only foreground model lifetimes require separate assumptions. CO2
pricing, emissions budgets, Pareto analysis and vintage improvements are later
extensions and are not part of this baseline.
